Testing how well QWEN R1 could conduct auto interp for SAE features

In [ ]:
# Load model quantized

import torch
from transformers import BitsAndBytesConfig, AutoModelForCausalLM, AutoTokenizer

config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

from transformers import AutoTokenizer, AutoModelForCausalLM
model_name_or_path = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B" # deepseek-ai/DeepSeek-R1-Distill-Qwen-1B

tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
model = AutoModelForCausalLM.from_pretrained(model_name_or_path, quantization_config=config, device_map="cuda:0")

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [2]:
model.eval()
auto_interp_prompt = [
    """
    Your task is to analyze and figure out the patern in the texts given later. It will contain tokens in brackets - <<example tokens>>. Create a label for the patern you recognize common in all of the highlighted texts in brackets. Use 20 words or less. 
    
    Texts to analyze:
1. and he was <<over the moon>> to find
2. we'll be laughing <<till the cows come home>>! Pro
3. thought Scotland was boring, but really there's more <<than meets the eye>>! I'd
 
    Please reason step by step, and put your final answer within \boxed{}
    "<think>\n
    """
]

In [ ]:
# TODO results need to be averaged
device = "cuda:0"
model_inputs = tokenizer(auto_interp_prompt, return_tensors="pt")
tokens = model_inputs["input_ids"].to(device)
attention_mask = model_inputs["attention_mask"].to(device)
    # Note the length of the input
input_length = tokens.shape[1]
generation_output = model.generate(
    tokens,
    attention_mask=attention_mask,
    max_new_tokens = 1000,
    # top_p = 0.9,
    temperature=0.6,
    # do_sample=True,
)
new_tokens = generation_output[0, input_length:].tolist()  # Get only the new token ids
# output = tokenizer.decode(generation_output[0])
output = tokenizer.decode(new_tokens, skip_special_tokens=True)

print(f'\n {output}')

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.



  Hi there! I'm looking at these sentences where certain words are highlighted in brackets. Let me try to figure out the pattern here.

First sentence: "and he was <<over the moon>> to find"  
Second sentence: "we'll be laughing <<till the cows come home>>! Pro"  
Third sentence: "thought Scotland was boring, but really there's more <<than meets the eye>>! I'd"  

Hmm, all the highlighted words seem to be describing emotions or states of being. "Over the moon" shows excitement, "till the cows come home" conveys something very positive, and "than meets the eye" indicates a surprising revelation. 

Each of these phrases is used to express a strong emotion or a significant change in perspective. They're all examples of idiomatic expressions that highlight a particular feeling or situation.

So, the common pattern here is that the highlighted words are expressions that emphasize strong emotions or surprising truths. They're all used to convey a sense of intensity or revelation in the sent

In [11]:
auto_interp_reconstruct_prompt = [
    """
    Your task is to analyze the texts later given. Output only all of text that you analyze and highlight words/tokens in brackets like so <<example text>> if they correspond to a certain feature.
    
    Feature to highlight: The pattern is that the highlighted words are idiomatic expressions emphasizing emotions or significant realizations.

    Texts to analyze:
1. and he was over the moon to find
2. we'll be laughing till the cows come home! Pro
3. thought Scotland was boring, but really there's more than meets the eye! I'd
 
    Reason step by step. Output the texts analyzed only, highlight the feature tokens in brackets like so <<example text>>.
    "<think>\n
    """
]

In [13]:
device = "cuda:0"
model_inputs = tokenizer(auto_interp_reconstruct_prompt, return_tensors="pt")
tokens = model_inputs["input_ids"].to(device)
attention_mask = model_inputs["attention_mask"].to(device)
    # Note the length of the input
input_length = tokens.shape[1]
generation_output = model.generate(
    tokens,
    attention_mask=attention_mask,
    max_new_tokens = 1000,
    # top_p = 0.9,
    temperature=0.6,
    # do_sample=True,
)
new_tokens = generation_output[0, input_length:].tolist()  # Get only the new token ids
# output = tokenizer.decode(generation_output[0])
output = tokenizer.decode(new_tokens, skip_special_tokens=True)

print(f'\n {output}')

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.



  I'm looking at the first text: "and he was over the moon to find". The phrase "over the moon" is an idiom that means being extremely happy or excited. It's expressing a strong emotion, which fits the feature.

     Moving to the second text: "we'll be laughing till the cows come home! Pro". The phrase "till the cows come home" is a common idiom meaning something will happen very soon. It doesn't convey a strong emotion or realization, so it doesn't fit the feature.

     Finally, the third text: "thought Scotland was boring, but really there's more than meets the eye! I'd". The phrase "there's more than meets the eye" is an idiom indicating that something is not what it seems. This reflects a significant realization, aligning with the feature.

     So, I'll highlight "over the moon" and "there's more than meets the eye" as they are idiomatic expressions emphasizing emotions or realizations.
</think>

1. and he was <<over the moon>> to find
2. we'll be laughing till the cows come ho

In [ ]:
# We could now compare % of how many tokens are reconstructed and compare against the ground truth
# Do need to keep in mind to punish highlighting more than the baseline as well

For comparison we load a dataframe of already annotated data from gpt-2 (Done by gpt-3.5-turbo?)
### TODO: we also need a coresponding error metric (based on the description did it fire correctly?), i.e. how good was the classiciation.

In [1]:
import pandas as pd
import requests
from sae_lens.toolkit.pretrained_saes_directory import get_pretrained_saes_directory

def get_autointerp_df(sae_release="gpt2-small-res-jb", sae_id="blocks.7.hook_resid_pre") -> pd.DataFrame:
    release = get_pretrained_saes_directory()[sae_release]
    neuronpedia_id = release.neuronpedia_id[sae_id]

    url = "https://www.neuronpedia.org/api/explanation/export?modelId={}&saeId={}".format(*neuronpedia_id.split("/"))
    headers = {"Content-Type": "application/json"}
    response = requests.get(url, headers=headers)

    data = response.json()
    return pd.DataFrame(data)


explanations_df = get_autointerp_df()
explanations_df.head()

,modelId,layer,index,description,explanationModelName,typeName
0,gpt2-small,7-res-jb,218,stars and dashed for censoring expletives,None,oai_token-act-pair
1,gpt2-small,7-res-jb,218,stars and dashes for censoring expletives,None,oai_token-act-pair
2,gpt2-small,7-res-jb,218,offensive language and expletives,None,oai_token-act-pair
3,gpt2-small,7-res-jb,2020,names of people,gpt-3.5-turbo,oai_token-act-pair
4,gpt2-small,7-res-jb,3493,references to Nazism,gpt-3.5-turbo,oai_token-act-pair
